<a href="https://colab.research.google.com/github/siri780/ab-test-criteo-uplift/blob/main/bigquery/criteo_bigquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Criteo A/B test in BigQuery (data warehouse version)
Loads the 14M-user Criteo uplift experiment into a free **BigQuery sandbox**, organizes it into warehouse layers
(raw → fact table → reporting views), and reproduces the key A/B results in BigQuery SQL.

**Before you start:** create a free project at https://console.cloud.google.com/bigquery (no credit card needed)
and paste its project ID below. Run the cells top to bottom (Runtime → Run all).

In [5]:
PROJECT_ID = "criteo-ab"
DATASET = "criteo"
LOCATION = "US"

## 1. Sign in to Google Cloud

In [6]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print('Connected to', client.project)

Connected to criteo-ab


## 2. Download the data (about 300 MB) and unzip it

In [7]:
!curl -sL -o criteo.csv.gz "https://huggingface.co/datasets/criteo/criteo-uplift/resolve/main/criteo-research-uplift-v2.1.csv.gz"
!gunzip -f criteo.csv.gz && ls -lh criteo.csv && head -3 criteo.csv

-rw-r--r-- 1 root root 3.1G Nov 11 01:59 criteo.csv
f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
12.616364906986497,10.059654474774547,8.976428838331028,4.679881620097284,10.280525225748212,4.115453421277861,0.29444271125560606,4.833814577796811,3.9553959684262416,13.190055934673358,5.300374864042156,-0.16867922100056124,1,0,0,0
12.616364906986497,10.059654474774547,9.002689495197341,4.679881620097284,10.280525225748212,4.115453421277861,0.29444271125560606,4.833814577796811,3.9553959684262416,13.190055934673358,5.300374864042156,-0.16867922100056124,1,0,0,0


## 3. Raw layer: load the CSV into a BigQuery table
The sandbox keeps tables for 60 days, which is plenty for this project. Loading takes a few minutes.

In [8]:
client.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
features = [bigquery.SchemaField(f"f{i}", "FLOAT64") for i in range(12)]
outcomes = [bigquery.SchemaField(c, "INT64") for c in ["treatment", "conversion", "visit", "exposure"]]
job_config = bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.CSV, skip_leading_rows=1,
                                    schema=features + outcomes, write_disposition="WRITE_TRUNCATE")
with open("criteo.csv", "rb") as f:
    job = client.load_table_from_file(f, f"{PROJECT_ID}.{DATASET}.raw_uplift", job_config=job_config)
job.result()
t = client.get_table(f"{PROJECT_ID}.{DATASET}.raw_uplift")
print(f"Loaded {t.num_rows:,} rows, {t.num_bytes/1e9:.2f} GB")

Loaded 13,979,592 rows, 1.79 GB


## 4. Helper: run SQL, show results and how much data each query scanned
The sandbox includes 1 TB of free queries per month; this whole notebook uses a few GB.

In [9]:
def sql(query):
    job = client.query(query.replace("{DS}", f"`{PROJECT_ID}.{DATASET}`"))
    df = job.to_dataframe()
    print(f"scanned {job.total_bytes_processed/1e6:,.0f} MB")
    return df

## 5. Fact layer: a clean, clustered user table
Adds a unique user key and clusters by group so per-group queries scan less data.

In [10]:
sql("""
CREATE OR REPLACE TABLE {DS}.fct_users
CLUSTER BY treatment AS
SELECT GENERATE_UUID() AS user_id,
       IF(treatment = 1, 'ads', 'no_ads') AS arm,
       treatment, exposure, visit, conversion,
       f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, f11
FROM {DS}.raw_uplift
""")

scanned 1,789 MB


""


## 6. Reporting layer: views analysts can query directly

In [11]:
sql("""
CREATE OR REPLACE VIEW {DS}.v_arm_rates AS
SELECT arm,
       COUNT(*) AS users,
       AVG(exposure) AS exposure_rate,
       AVG(visit) AS visit_rate,
       AVG(conversion) AS conversion_rate
FROM {DS}.fct_users
GROUP BY arm
""")
sql("""
CREATE OR REPLACE VIEW {DS}.v_lift AS
WITH r AS (SELECT * FROM {DS}.v_arm_rates),
t AS (SELECT * FROM r WHERE arm = 'ads'), c AS (SELECT * FROM r WHERE arm = 'no_ads')
SELECT metric, treat_rate, control_rate,
       treat_rate - control_rate AS abs_lift,
       (treat_rate - control_rate) - 1.96 * se AS abs_lift_lo,
       (treat_rate - control_rate) + 1.96 * se AS abs_lift_hi,
       treat_rate / control_rate - 1 AS rel_lift
FROM (
  SELECT 'visit' AS metric, t.visit_rate AS treat_rate, c.visit_rate AS control_rate,
         SQRT(t.visit_rate*(1-t.visit_rate)/t.users + c.visit_rate*(1-c.visit_rate)/c.users) AS se FROM t, c
  UNION ALL
  SELECT 'conversion', t.conversion_rate, c.conversion_rate,
         SQRT(t.conversion_rate*(1-t.conversion_rate)/t.users + c.conversion_rate*(1-c.conversion_rate)/c.users) FROM t, c
)
""")

scanned 0 MB
scanned 0 MB


""


## 7. Results
**Sample ratio check:** the design is 85% ads / 15% no ads.

In [12]:
sql("SELECT arm, users, ROUND(100 * users / SUM(users) OVER (), 3) AS pct FROM {DS}.v_arm_rates ORDER BY arm")

scanned 76 MB


,arm,users,pct
0,ads,11882655,85.0
1,no_ads,2096937,15.0


**Lift with 95% confidence intervals** (should match the Python analysis: conversions +59%).

In [13]:
sql("SELECT * FROM {DS}.v_lift")

scanned 300 MB


,metric,treat_rate,control_rate,abs_lift,abs_lift_lo,abs_lift_hi,rel_lift
0,conversion,0.003089,0.001938,0.001152,0.001085,0.001219,0.594488
1,visit,0.048543,0.038201,0.010342,0.010056,0.010629,0.270737


**Who responds most?** Conversion lift by decile of feature f0 (deciles from approximate quantiles, which scale to billions of rows).

In [14]:
sql("""
WITH q AS (SELECT APPROX_QUANTILES(f0, 10) AS cuts FROM {DS}.fct_users),
d AS (SELECT arm, conversion, LEAST(GREATEST(RANGE_BUCKET(f0, cuts), 1), 10) AS decile FROM {DS}.fct_users, q)
SELECT decile,
       AVG(IF(arm = 'ads', conversion, NULL)) AS ads_rate,
       AVG(IF(arm = 'no_ads', conversion, NULL)) AS no_ads_rate,
       SAFE_DIVIDE(AVG(IF(arm = 'ads', conversion, NULL)), AVG(IF(arm = 'no_ads', conversion, NULL))) - 1 AS rel_lift
FROM d GROUP BY decile ORDER BY decile
""")

scanned 300 MB


,decile,ads_rate,no_ads_rate,rel_lift
0,3,0.004375,0.002771,0.578915
1,4,0.010328,0.006154,0.678285
2,5,0.004133,0.002739,0.508866
3,6,0.000667,0.000446,0.496713
4,7,0.001047,0.000812,0.290234
5,8,0.000503,0.000401,0.253351
6,9,0.000596,0.000471,0.264990
7,10,0.000308,0.000254,0.213565


## 8. Clean up local files
The BigQuery tables stay in your sandbox for 60 days.

In [15]:
!rm -f criteo.csv && echo removed local copy

removed local copy
